In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

batch_size = 512
learning_rate = 0.1
num_epoch = 10

print("Device:", device)

In [ ]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
])

train_dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    transform=transform,
    download=True,
)

test_dataset = datasets.FashionMNIST(
    root="./data",
    train=False,
    transform=transform,
    download=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

print("Training examples:", len(train_dataset))
print("Test examples:", len(test_dataset))

In [ ]:
def softmax_from_scratch(logits):
    shifted_logits = (
        logits
        - logits.max(dim=-1, keepdim=True).values
    )

    exponentials = shifted_logits.exp()

    return (
        exponentials
        / exponentials.sum(dim=-1, keepdim=True)
    )


def cross_entropy_from_probabilities(
    probabilities,
    labels,
):
    batch_indices = torch.arange(
        probabilities.shape[0],
        device=probabilities.device,
    )

    # 정답 라벨의 확률만 골라내기
    correct_probabilities = probabilities[
        batch_indices,
        labels,
    ]

    return -(
        correct_probabilities
        .clamp_min(1e-12)
        .log()
    ).mean()


class SoftmaxRegressionScratch:
    def __init__(
        self,
        num_inputs,
        num_outputs,
        sigma=0.01,
    ):
        self.weights = (
            torch.randn(
                num_inputs,
                num_outputs,
                device=device,
            )
            * sigma
        ).requires_grad_()

        self.bias = torch.zeros(
            num_outputs,
            device=device,
            requires_grad=True,
        )

    def parameters(self):
        return [
            self.weights,
            self.bias,
        ]

    def __call__(self, images):
        flattened = images.reshape(
            images.shape[0],
            -1,
        )

        logits = (
            flattened @ self.weights
            + self.bias
        )

        return softmax_from_scratch(logits)

In [ ]:
@torch.no_grad()
def evaluate(model, data_loader):
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    
    for images, labels in data_loader:
        images = images.to(
            device,
            non_blocking=True,
        )
        labels = labels.to(
            device,
            non_blocking=True,
        )
        
        probabilities = model(images)
        
        loss = cross_entropy_from_probabilities(
            probabilities,
            labels
        )
        
        predictions = probabilities.argmax(
            dim=-1
        )
        
        current_batch_size = labels.shape[0]
        
        # 배치 평균 loss를 데이터 개수만큼 다시 확장한다.
        total_loss += (
            loss.item()
            * current_batch_size
        )

        total_correct += (
            predictions == labels
        ).sum().item()

        total_examples += current_batch_size

    return {
        "loss": total_loss / total_examples,
        "accuracy": total_correct / total_examples,
    }

In [ ]:
model = SoftmaxRegressionScratch(
    num_inputs=32 * 32,
    num_outputs=10,
)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=learning_rate,
)

for epoch in range(num_epoch):
    for images, labels in train_loader:
        images = images.to(
            device,
            non_blocking=True,
        )
        labels = labels.to(
            device,
            non_blocking=True,
        )
        
        # softmax
        probabilities = model(images)
        
        loss = cross_entropy_from_probabilities(
            probabilities,
            labels,
        )
        
        # 이전 미니배치의 gradient를 제거
        optimizer.zero_grad()
        
        # 현재 loss의 gradient를 계산
        loss.backward()
        
        # weights와 bias를 갱신
        optimizer.step()
    
    train_metrics = evaluate(
        model,
        train_loader,
    )

    validation_metrics = evaluate(
        model,
        test_loader,
    )
    
    print(
        f"Epoch {epoch + 1:2d} | "
        f"train loss {train_metrics['loss']:.4f} | "
        f"train acc {train_metrics['accuracy']:.4f} | "
        f"val loss {validation_metrics['loss']:.4f} | "
        f"val acc {validation_metrics['accuracy']:.4f}"
    )